In [ ]:
!pip install datasets tiktoken -q

In [ ]:
import os, numpy as np, tiktoken
from datasets import load_dataset

In [ ]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

TRAIN_TARGET  = 400_000_000
VAL_TARGET    =  20_000_000
TRAIN_BIN     = "/kaggle/working/train.bin"
VAL_BIN       = "/kaggle/working/val.bin"

In [ ]:
if os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN) \
   and os.path.getsize(TRAIN_BIN) > 0 and os.path.getsize(VAL_BIN) > 0:
    print("Binary files exist, skipping tokenization.")
else:
    enc = tiktoken.get_encoding("gpt2")
    train_tokens, val_tokens = [], []
    print("Streaming FineWeb-Edu...")
    dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT",
                           split="train", streaming=True, trust_remote_code=True)
    for i, example in enumerate(dataset):
        tokens = enc.encode(example["text"])
        if len(train_tokens) < TRAIN_TARGET:
            train_tokens.extend(tokens)
        elif len(val_tokens) < VAL_TARGET:
            val_tokens.extend(tokens)
        else:
            break
        if i % 5000 == 0:
            print(f"  {len(train_tokens)/1e6:.1f}M train | {len(val_tokens)/1e6:.1f}M val")

    np.array(train_tokens, dtype=np.uint16).tofile(TRAIN_BIN)
    np.array(val_tokens,   dtype=np.uint16).tofile(VAL_BIN)
    print("saved.")

In [ ]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")
print(f"train: {len(train_data):,} | val: {len(val_data):,}")
assert train_data.max() < 50257, f"OOB token in train: {train_data.max()}"
assert val_data.max()   < 50257, f"OOB token in val: {val_data.max()}"
print("tokens OK")